<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-01-classical-ml-statsrefresher/probability-statistics/notebooks/exercises-0_5_4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎲 Lesson 0.5.4: Probability for AI Engineers — Practice Exercises

**Netsetos GenAI Course** | Module 0.5 — Classical ML & Stats Refresher

6 exercises covering discrete PMFs, continuous PDFs, temperature-controlled token sampling, log-normal latency modeling, spectrograms, and a full probability dashboard.

**Key insight:** Token prediction = categorical (discrete). Latency = log-normal (continuous). Always use percentiles, never means, for skewed distributions.

---

## Exercise 1 (Easy): Discrete Distributions — PMF Builder

### 📋 Task
1. Build PMFs for Bernoulli, Binomial, Poisson, Categorical
2. Verify each sums to 1.0
3. 10K samples, compare empirical vs theoretical

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 1
import numpy as np
from scipy import stats
np.random.seed(42)

# TODO: build PMFs, verify, sample, compare

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
from scipy import stats
np.random.seed(42)

print('=== Discrete Distributions ===')

# 1. Bernoulli
p = 0.3
pmf_bern = [1-p, p]
print(f'\nBernoulli(p={p}): P(0)={1-p:.3f}, P(1)={p:.3f}, sum={sum(pmf_bern):.3f}')
samples = np.random.binomial(1, p, 10000)
print(f'  Empirical: P(0)={np.mean(samples==0):.3f}, P(1)={np.mean(samples==1):.3f}')

# 2. Binomial
n, p = 10, 0.3
k = np.arange(0, n+1)
pmf_binom = stats.binom.pmf(k, n, p)
print(f'\nBinomial(n={n}, p={p}): sum={pmf_binom.sum():.6f}')
samples = np.random.binomial(n, p, 10000)
for val in [0, 1, 2, 3, 4, 5]:
    print(f'  k={val}: theory={pmf_binom[val]:.3f} emp={np.mean(samples==val):.3f}')

# 3. Poisson
lam = 4
k = np.arange(0, 15)
pmf_pois = stats.poisson.pmf(k, lam)
print(f'\nPoisson(λ={lam}): sum={pmf_pois.sum():.6f}')
samples = np.random.poisson(lam, 10000)
for val in [0, 2, 4, 6, 8]:
    print(f'  k={val}: theory={stats.poisson.pmf(val,lam):.3f} emp={np.mean(samples==val):.3f}')

# 4. Categorical (token prediction!)
vocab = ['the', 'cat', 'sat', 'on', 'mat', 'dog']
logits = np.array([2.1, 3.5, 0.8, 1.2, -0.3, 0.5])
pmf_cat = np.exp(logits) / np.exp(logits).sum()
print(f'\nCategorical (softmax): sum={pmf_cat.sum():.6f}')
for t, p in sorted(zip(vocab, pmf_cat), key=lambda x: -x[1]):
    bar = '█' * int(p * 30)
    print(f'  {t:6s} {p:.4f} {bar}')

print('\n✅ All PMFs sum to 1.0')

</details>

---

## Exercise 2 (Easy): Continuous Distributions — PDF + Percentiles

### 📋 Task
1. 10K samples from Gaussian, Log-Normal, Exponential, Beta
2. Mean, median, p50/p95/p99
3. Show mean-median gap for skewed distributions

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 2
import numpy as np
np.random.seed(42)

# TODO: 4 distributions, percentiles, mean-median gap

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
np.random.seed(42)

dists = {
    'Gaussian':    np.random.randn(10000),
    'Log-Normal':  np.random.lognormal(5.5, 0.6, 10000),
    'Exponential': np.random.exponential(200, 10000),
    'Beta':        np.random.beta(2, 5, 10000),
}

print(f'{"Dist":<14} {"Mean":>8} {"Median":>8} {"Gap":>8} {"p95":>8} {"p99":>8} {"Skewed?"}')
print('-' * 72)
for name, s in dists.items():
    mean, med = s.mean(), np.median(s)
    gap = mean - med
    p95, p99 = np.percentile(s, 95), np.percentile(s, 99)
    skewed = 'YES → use percentiles' if abs(gap) > abs(mean)*0.05 else 'No (symmetric)'
    print(f'  {name:<12} {mean:>7.2f} {med:>7.2f} {gap:>7.2f} {p95:>7.2f} {p99:>7.2f}  {skewed}')

print('\nRule: For right-skewed distributions (latency!), ALWAYS use percentiles, not mean.')

</details>

---

## Exercise 3 (Medium): Token Prediction as Categorical Distribution

### 📋 Task
1. 20-word vocab with logits → softmax → PMF
2. Temperature sampling at 0.1, 0.5, 1.0, 2.0
3. Entropy at each temperature
4. Generate 20-token sequences

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 3
import numpy as np
np.random.seed(42)

vocab = [f'word_{i}' for i in range(20)]
logits = np.random.randn(20) * 2

# TODO: softmax, temperature, entropy, generate

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
np.random.seed(42)

vocab = [f'w{i}' for i in range(20)]
logits = np.random.randn(20) * 2

def softmax_t(logits, T):
    scaled = logits / max(T, 1e-8)
    e = np.exp(scaled - np.max(scaled))
    return e / e.sum()

def entropy(p):
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

print(f'{"Temp":>5} {"Entropy":>10} {"Top-1 prob":>11} {"Top-3 coverage":>15}')
print('-' * 46)
for T in [0.1, 0.5, 1.0, 2.0]:
    pmf = softmax_t(logits, T)
    h = entropy(pmf)
    top1 = pmf.max()
    top3 = np.sort(pmf)[-3:].sum()
    print(f'  {T:>3.1f} {h:>9.2f} bits {top1:>10.4f} {top3:>14.3f}')

# Generate sequences
np.random.seed(42)
print('\nGenerated sequences (20 tokens):')
for T in [0.1, 1.0, 2.0]:
    pmf = softmax_t(logits, T)
    seq = [np.random.choice(vocab, p=pmf) for _ in range(20)]
    print(f'  T={T}: {" ".join(seq[:10])}...')

print('\nT↑ → entropy↑ → more randomness → less repetitive')

</details>

---

## Exercise 4 (Medium): Log-Normal Latency Modeling

### 📋 Task
1. 10K API calls with log-normal latency
2. p50/p95/p99, show mean vs percentile gap
3. SLO checker: p99 < 1000ms
4. Add 2% spikes, re-check

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 4
import numpy as np
np.random.seed(42)

latencies = np.random.lognormal(mean=5.5, sigma=0.6, size=10000)

# TODO: stats, SLO check, add spikes, re-check

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
np.random.seed(42)

latencies = np.random.lognormal(mean=5.5, sigma=0.6, size=10000)

def report(data, label):
    mean, med = data.mean(), np.median(data)
    p50, p95, p99 = [np.percentile(data, p) for p in [50, 95, 99]]
    slo = p99 < 1000
    mean_alert = mean < 500
    print(f'\n--- {label} ---')
    print(f'  Mean: {mean:.0f}ms, Median: {med:.0f}ms (gap: {mean-med:.0f}ms)')
    print(f'  p50={p50:.0f}ms  p95={p95:.0f}ms  p99={p99:.0f}ms')
    print(f'  SLO (p99 < 1000ms): {"PASS ✅" if slo else "FAIL ❌"}')
    print(f'  Mean alert (< 500ms): {"PASS" if mean_alert else "FAIL"}')
    if mean_alert and not slo:
        print(f'  ⚠️ Mean-based alert MISSED the p99 violation!')
    return slo

report(latencies, 'Normal conditions')

# Add 2% spikes
spike_lat = latencies.copy()
idx = np.random.choice(len(spike_lat), int(len(spike_lat)*0.02), replace=False)
spike_lat[idx] *= 5
report(spike_lat, 'With 2% latency spikes')

print('\n→ Mean barely changed, but p99 blew up. ALWAYS use percentiles for latency SLOs.')

</details>

---

## Exercise 5 (Hard): Spectrogram from Scratch

### 📋 Task
1. Generate 1s audio: 3 frequencies + noise at 16kHz
2. STFT: 25ms windows, 10ms hop, Hanning window
3. Magnitude spectrogram
4. Mel-scale binning (40 bands)
5. Print shape and dominant frequencies

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 5
import numpy as np

sr = 16000
t = np.arange(0, 1.0, 1/sr)
signal = (np.sin(2*np.pi*440*t) +
          0.5*np.sin(2*np.pi*880*t) +
          0.3*np.sin(2*np.pi*1320*t))
signal += np.random.randn(len(t)) * 0.1

# TODO: STFT, spectrogram, mel binning

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np

sr = 16000
t = np.arange(0, 1.0, 1/sr)
signal = (np.sin(2*np.pi*440*t) +
          0.5*np.sin(2*np.pi*880*t) +
          0.3*np.sin(2*np.pi*1320*t))
signal += np.random.randn(len(t)) * 0.1

# STFT parameters
win_size = int(0.025 * sr)  # 400 samples = 25ms
hop_size = int(0.010 * sr)  # 160 samples = 10ms
n_frames = (len(signal) - win_size) // hop_size + 1

# Build spectrogram
spectrogram = []
for i in range(n_frames):
    frame = signal[i*hop_size : i*hop_size + win_size]
    windowed = frame * np.hanning(win_size)
    fft_mag = np.abs(np.fft.rfft(windowed))
    spectrogram.append(fft_mag)

spectrogram = np.array(spectrogram)
print(f'Spectrogram shape: {spectrogram.shape}')
print(f'  = {n_frames} time frames × {spectrogram.shape[1]} frequency bins')

# Dominant frequencies
freqs = np.fft.rfftfreq(win_size, 1/sr)
for frame_idx in [0, n_frames//2, n_frames-1]:
    top3_bins = spectrogram[frame_idx].argsort()[-3:][::-1]
    top3_freqs = freqs[top3_bins]
    print(f'  Frame {frame_idx}: dominant = {top3_freqs.astype(int)} Hz')

# Simple mel binning (40 bands)
n_mels = 40
fft_bins = spectrogram.shape[1]
bins_per_mel = fft_bins // n_mels
mel_spec = np.zeros((n_frames, n_mels))
for m in range(n_mels):
    start = m * bins_per_mel
    end = min(start + bins_per_mel, fft_bins)
    mel_spec[:, m] = spectrogram[:, start:end].mean(axis=1)

# Log scale (as Whisper does)
log_mel = np.log(mel_spec + 1e-10)
print(f'\nMel spectrogram: {log_mel.shape}')
print(f'  = {n_frames} frames × {n_mels} mel bands')
print(f'  This is Whisper\'s input format!')

</details>

---

## Exercise 6 (Hard): Full Probability Pipeline

### 📋 Task
1. Categorical: 1000 token predictions, entropy, top-k coverage
2. Gaussian: 768-dim embedding, verify N(0,1)
3. Log-Normal: 1000 latencies, p50/p95/p99, SLO
4. Poisson: hourly errors, P(>5)
5. Beta: A/B test, P(B>A)
6. Unified dashboard

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 6
import numpy as np
from scipy import stats
np.random.seed(42)

# TODO: all 5 distributions + dashboard

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np
from scipy import stats
np.random.seed(42)

print('═' * 50)
print('  PRODUCTION PROBABILITY DASHBOARD')
print('═' * 50)

# 1. Categorical (token generation)
logits = np.random.randn(1000) * 2
pmf = np.exp(logits) / np.exp(logits).sum()
H = -np.sum(pmf[pmf>0] * np.log2(pmf[pmf>0]))
top5_coverage = np.sort(pmf)[-5:].sum()
print(f'\n1. Token Generation (Categorical):')
print(f'   Entropy: {H:.1f} bits, Top-5 coverage: {top5_coverage*100:.1f}%')

# 2. Gaussian (embedding)
emb = np.random.randn(768)
print(f'\n2. Embedding (Gaussian):')
print(f'   Mean: {emb.mean():.3f}, Std: {emb.std():.3f}')
print(f'   Looks N(0,1): {"✅" if abs(emb.mean()) < 0.1 and abs(emb.std()-1) < 0.1 else "❌"}')

# 3. Log-Normal (latency)
lat = np.random.lognormal(5.5, 0.6, 1000)
p50, p95, p99 = [np.percentile(lat, p) for p in [50, 95, 99]]
slo = p99 < 1000
print(f'\n3. API Latency (Log-Normal):')
print(f'   p50={p50:.0f}ms  p95={p95:.0f}ms  p99={p99:.0f}ms')
print(f'   SLO (p99 < 1000ms): {"PASS ✅" if slo else "FAIL ❌"}')

# 4. Poisson (errors)
errors = np.random.poisson(3, 1000)
p_high = np.mean(errors > 5)
print(f'\n4. Error Rate (Poisson, λ=3):')
print(f'   Mean: {errors.mean():.2f}/hr, P(>5 errors): {p_high:.3f}')
print(f'   Alert (>5/hr happens >10%): {"⚠️ WARNING" if p_high > 0.1 else "OK ✅"}')

# 5. Beta (A/B test)
posterior_a = stats.beta(15+1, 185+1)  # 15 clicks / 200
posterior_b = stats.beta(22+1, 178+1)  # 22 clicks / 200
p_b_wins = np.mean(posterior_b.rvs(10000) > posterior_a.rvs(10000))
print(f'\n5. A/B Test (Beta):')
print(f'   A: 15/200 = 7.5%, B: 22/200 = 11.0%')
print(f'   P(B > A) = {p_b_wins:.3f}')
print(f'   Confidence: {"HIGH" if p_b_wins > 0.9 else "MODERATE" if p_b_wins > 0.8 else "LOW"}')

print(f'\n{"═" * 50}')
print(f'  All systems nominal ✅' if slo else '  ⚠️ SLO violation detected')
print(f'{"═" * 50}')

</details>

---

## 🎉 Done!

You've mastered every probability distribution used in production GenAI:
- **Categorical** — token prediction (softmax = PMF)
- **Gaussian** — embedding space, weight initialization
- **Log-Normal** — API latency (ALWAYS use percentiles!)
- **Poisson** — error rate monitoring
- **Beta** — Bayesian A/B testing
- **Spectrogram** — signal processing for Whisper

These are the statistical foundations for **Modules 2-11.** Module 0.5 complete!